# 道琼斯 30：交互式 Brinson 归因

本教程用本地 `brinsons` 和 Yahoo Finance 数据重现 [MathWorks 的 Brinson 分析流程](https://www.mathworks.com/help/releases/R2023a/finance/workflow-for-performance-attribution-using-brinson-model.html)。请从上到下运行各单元。

保存的示例覆盖 12 个完整月份，组合权重为**模拟值**，基准每月重新设为**等权**。该基准不是按价格加权的道琼斯指数。历史月份使用当前成分股与 Yahoo 当前行业分类，因此不是历史成分股回测；结果未计入费用和交易。

## 1. 环境与数据

如需安装依赖，在当前 Jupyter 内核对应环境运行 `python -m pip install -e . ipywidgets matplotlib jupyter`。可从项目根目录或 `tutorials/zh/` 启动 Jupyter。默认读取本地快照，不发起网络请求。

如需下载新快照，先从项目根目录执行 `python scripts/fetch_dow30.py`，再执行 `python scripts/dow30_brinson_analysis.py`。这会替换生成的 CSV 数据；之后重新运行本教程。

In [ ]:
from pathlib import Path
import sys
import polars as pl
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import ipywidgets as widgets
from IPython.display import display

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "brinsons").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Start Jupyter inside the brinson project directory.")
sys.path.insert(0, str(ROOT / "src"))
from brinsons import calculate_multiple_period, bhb, fachler, link_carino, link_grap, link_frongello, link_menchero, link_geometric, chart_multiple_period_excess_return_waterfall

DATA = ROOT / "data" / "dow30_brinson"
asset = pl.read_csv(DATA / "asset_table.csv").sort(["Period", "Name"])
periods = pl.read_csv(DATA / "periods.csv", try_parse_dates=True).sort("Period")
prices = pl.read_csv(DATA / "monthly_prices.csv", try_parse_dates=True)
print(f"{asset['Name'].n_unique()} stocks · {periods.height} periods · {asset['Category'].n_unique()} sectors")
display(periods)

## 2. 准备个股表

表结构与 MathWorks 示例一致：`Period`、`Name`、`Return`、`Category`、`PortfolioWeight`、`BenchmarkWeight`。收益率是调整后收盘价计算的月收益。组合目标权重及等权基准每月重置；演示用目标权重每三期调整一次。所有数值均为十进制。

In [ ]:
expected = ["Period", "Name", "Return", "Category", "PortfolioWeight", "BenchmarkWeight"]
assert asset.columns == expected
price_check = prices.join(periods.select("Period", pl.col("PeriodEnd").alias("date")), on="date").rename({"ticker": "Name"})
joined = asset.join(price_check, on=["Period", "Name"])
assert joined.height == asset.height
assert joined.select((pl.col("Return") - (pl.col("close") / pl.col("previous_close") - 1)).abs().max()).item() < 1e-12
weights = asset.group_by("Period").agg(pl.col("PortfolioWeight", "BenchmarkWeight").sum()).sort("Period")
assert weights.select(((pl.col("PortfolioWeight") - 1).abs() < 1e-10).all()).item()
assert weights.select(((pl.col("BenchmarkWeight") - 1).abs() < 1e-10).all()).item()
display(asset.head(12))

## 3. 计算归因

按期间和 Yahoo 行业汇总个股权重及加权贡献，再用 `brinsons` 计算归因。选股效应比较行业内组合与基准持仓。算术 linking 对账到复合后的收益之差；几何 linking 满足另一种乘法恒等式。

科技行业乘数只调整示例的**每月初始目标权重**，随后按期重新归一化。它是利用既有收益作情景分析，并不代表真实交易策略。

In [ ]:
ATTRIBUTION = {"Brinson–Fachler": fachler, "Brinson–Hood–Beebower": bhb}
LINKING = {"Menchero": link_menchero, "Cariño": link_carino, "GRAP": link_grap,
           "Frongello": link_frongello, "Geometric": link_geometric}

def calculate(technology_multiplier=1.0, attribution_method="Brinson–Fachler", linking_method="Menchero"):
    adjusted = asset.with_columns(
        (pl.col("PortfolioWeight") * pl.when(pl.col("Category") == "Technology")
         .then(technology_multiplier).otherwise(1.0)).alias("_weight")
    ).with_columns((pl.col("_weight") / pl.col("_weight").sum().over("Period")).alias("PortfolioWeight"))
    category = adjusted.group_by(["Period", "Category"]).agg(
        pl.col("PortfolioWeight").sum().alias("portfolio_weight"),
        pl.col("BenchmarkWeight").sum().alias("benchmark_weight"),
        (pl.col("PortfolioWeight") * pl.col("Return")).sum().alias("p_contribution"),
        (pl.col("BenchmarkWeight") * pl.col("Return")).sum().alias("b_contribution"),
    ).with_columns(
        (pl.col("p_contribution") / pl.col("portfolio_weight")).alias("portfolio_return"),
        (pl.col("b_contribution") / pl.col("benchmark_weight")).alias("benchmark_return"),
    ).rename({"Period": "period", "Category": "segment"}).sort(["period", "segment"])
    workflow = calculate_multiple_period(category.select(
        "period", "segment", "portfolio_weight", "benchmark_weight", "portfolio_return", "benchmark_return"),
        attribution=ATTRIBUTION[attribution_method], linking=LINKING[linking_method])
    result, linked = workflow.periods, workflow.linked
    if linking_method == "Geometric":
        total = linked.summary.row(0, named=True)
        assert abs((1 + total["allocation"]) * (1 + total["selection"]) - 1 - total["excess_return"]) < 1e-10
        sector_effects = None
    else:
        total = linked.summary.row(0, named=True)
        assert abs(total["allocation"] + total["selection"] + total["interaction"] - total["excess_return"]) < 1e-10
        sector_effects = linked.detail.group_by("segment").agg(
            pl.col("linked_allocation", "linked_selection", "linked_interaction").sum()
        ).with_columns(pl.sum_horizontal("linked_allocation", "linked_selection", "linked_interaction").alias("active_effect")).sort("segment")
    return adjusted, category, result, linked, sector_effects

adjusted, category, result, linked, sector_effects = calculate()
print("全区间归因（十进制收益）— Linking: Menchero")
display(linked.summary)
print("各期行业结果")
display(result.detail.head(12))

## 4. 对比行业收益、权重和归因

下列行业收益由每月行业条件收益复合而来，各月行业内部权重会重置。行业效应采用 Menchero 算术链接，不等同于行业收益。

In [ ]:
category_summary = category.group_by("segment").agg(
    pl.col("portfolio_weight", "benchmark_weight").mean(),
    ((1 + pl.col("portfolio_return")).product() - 1).alias("portfolio_category_return"),
    ((1 + pl.col("benchmark_return")).product() - 1).alias("benchmark_category_return"),
).join(sector_effects, on="segment").sort("segment")
display(category_summary)
labels = category_summary["segment"].to_list()
y = list(range(len(labels)))
fig, axes = plt.subplots(1, 3, figsize=(19, 7), layout="constrained")
for ax, pcol, bcol, title in [
    (axes[0], "portfolio_category_return", "benchmark_category_return", "行业复合收益 — Linking: none"),
    (axes[1], "portfolio_weight", "benchmark_weight", "行业月均权重 — Linking: none"),
]:
    ax.barh([i - .18 for i in y], category_summary[pcol].to_list(), height=.36, label="Portfolio", color="#2366a8")
    ax.barh([i + .18 for i in y], category_summary[bcol].to_list(), height=.36, label="Benchmark", color="#aab5c5")
    ax.set_title(title); ax.set_yticks(y, labels); ax.invert_yaxis(); ax.xaxis.set_major_formatter(PercentFormatter(1)); ax.legend(fontsize=8)
axes[2].barh(y, category_summary["active_effect"].to_list(), color="#3f8d70")
axes[2].set(title="行业主动效应 — Linking: Menchero", yticks=y, yticklabels=labels)
axes[2].invert_yaxis(); axes[2].xaxis.set_major_formatter(PercentFormatter(1))
for ax in axes:
    ax.axvline(0, color="#666666", lw=.7); ax.grid(axis="x", color="#e6e9ed"); ax.set_axisbelow(True)
plt.show()

## 5. 交互探索

调整科技行业乘数、Brinson 方法或 linking 方法，重新计算完整区间；选择行业与期间查看各月权重、收益和效应。几何 linking 报告几何超额收益，两项效应按乘法合成；本库不为其生成可加的行业链接效应。

总体超额收益瀑布图随所选 linking 方法变化。几何版本展示配置、选股、两者交叉项和相对超额收益。

In [ ]:
method_control = widgets.Dropdown(options=list(ATTRIBUTION), value="Brinson–Fachler", description="归因:")
link_control = widgets.Dropdown(options=list(LINKING), value="Menchero", description="链接:")
tilt_control = widgets.FloatSlider(value=1.0, min=0.5, max=1.5, step=0.05, description="科技倾斜:", continuous_update=False, readout_format=".2f")
period_control = widgets.IntSlider(value=1, min=int(periods["Period"].min()), max=int(periods["Period"].max()), step=1, description="期间:", continuous_update=False)
sector_control = widgets.Dropdown(options=sorted(asset["Category"].unique().to_list()), description="行业:")

def explore(attribution_method, linking_method, technology_multiplier, period, sector):
    _, cats, per, linked_result, effects = calculate(technology_multiplier, attribution_method, linking_method)
    total = linked_result.summary.row(0, named=True)
    print(f"多期 linking 方法: {linking_method}")
    kind = "几何超额收益" if linking_method == "Geometric" else "算术超额收益"
    print(f"Portfolio: {total['portfolio_return']:.4%}   Benchmark: {total['benchmark_return']:.4%}   {kind}: {total['excess_return']:.4%}")
    print(f"Allocation: {total['allocation']:.4%}   Selection: {total['selection']:.4%}   Interaction: {total['interaction']:.4%}")
    print("所选行业与期间（十进制收益和权重）:")
    display(per.detail.filter((pl.col("period") == period) & (pl.col("segment") == sector)).select(
        "period", "segment", "portfolio_weight", "benchmark_weight", "portfolio_return", "benchmark_return", "allocation", "selection", "interaction"))
    timeline = cats.filter(pl.col("segment") == sector).sort("period")
    fig, ax = plt.subplots(figsize=(9, 3.2), layout="constrained")
    ax.plot(timeline["period"].to_list(), timeline["portfolio_return"].to_list(), marker="o", label="Portfolio")
    ax.plot(timeline["period"].to_list(), timeline["benchmark_return"].to_list(), marker="o", label="Benchmark")
    ax.axvline(period, color="#888888", linestyle="--", linewidth=.8)
    ax.axhline(0, color="#777777", linewidth=.7); ax.yaxis.set_major_formatter(PercentFormatter(1))
    ax.set(title=f"{sector}: periodic sector returns — Linking: none", xlabel="Period", ylabel="Return")
    ax.grid(alpha=.2); ax.legend(); plt.show()
    fig, ax = chart_multiple_period_excess_return_waterfall(
        per, by_category=False, linking=LINKING[linking_method])
    plt.show()
    if effects is not None:
        print(f"行业链接效应（十进制收益）— Linking: {linking_method}")
        display(effects.filter(pl.col("segment") == sector))

controls = widgets.VBox([widgets.HBox([method_control, link_control]), tilt_control,
                        widgets.HBox([period_control, sector_control])])
output = widgets.interactive_output(explore, {"attribution_method": method_control, "linking_method": link_control,
    "technology_multiplier": tilt_control, "period": period_control, "sector": sector_control})
display(controls, output)

## 数据来源与解读

- 输入来自本地脚本生成的 Yahoo Finance 调整后收盘价与 Yahoo 行业标签快照。
- 分析工作簿和详细方法：`outputs/dow30_brinson/Dow30_brinson_analysis.xlsx` 与 `outputs/dow30_brinson/analysis.md`。
- 归因可选 Brinson–Fachler 或 BHB；linking 在上方明确选择。本实现的 GRAP 与 Frongello 使用等价系数。
- 行业分类和成分股为当前快照；结果仅为假设示例，不能据此推断投资能力。